# 09 Appendix — Supporting Analyses

Supporting material for `09_analysis.ipynb`. Not part of the main narrative.

**Requires:** run `09_analysis.ipynb` first (or `09_run_orchestrator.ipynb`) to populate `tables_n17/`.

- **§A** — Female alter betweenness (descriptive extension)
- **§B** — Directed+weighted keystone experiment (STATUS: EXPERIMENT)

## Setup

In [1]:
from pathlib import Path
import sys, pandas as pd, numpy as np, networkx as nx
from scipy import stats

CLEAN = Path('..').resolve()
sys.path.insert(0, str(CLEAN / 'analysis' / 'h1_homophily'))
TBL  = CLEAN / 'analysis' / 'h1_homophily' / 'tables_n17'
DATA = CLEAN / 'data' / '02_processed'

from _common import GENDER_PALETTE, set_style, cliffs_delta, mannwhitney, perm_test_diff, bootstrap_diff_ci, power_note
from phase3_crossfilm_method_validation import load_character_meta, build_id_remap, gender_map, protag_id
set_style()

DROP_FILMS  = {'soul_2020'}
DROP_PROTAG = ('monsters_inc_2001', 'Mike')
def apply_conventions(d):
    d = d[~d['film_id'].isin(DROP_FILMS)].copy()
    d = d[~((d['film_id']==DROP_PROTAG[0]) & (d['protagonist']==DROP_PROTAG[1]))].copy()
    return d.reset_index(drop=True)

print('Setup OK')

Setup OK


---
## Section A - Female Alter Betweenness (descriptive)

**Metric:** `female_alter_betw_z` - mean betweenness z-score of the protagonist's *female* alters.
Positive = female alters are structurally load-bearing in the network.

**Source:** `step4_female_alter_betw_z.csv` (generated by orchestrator from `betweenness_null.csv` per film).

In [2]:
fa = pd.read_csv(TBL / 'step4_female_alter_betw_z.csv')
fa = apply_conventions(fa)

F_v = fa[fa['lead_gender']=='F']['female_alter_betw_z'].dropna().values
M_v = fa[fa['lead_gender']=='M']['female_alter_betw_z'].dropna().values

print(f'n_F={len(F_v)}  n_M={len(M_v)}')
print(f'Median F={np.median(F_v):+.3f}  Median M={np.median(M_v):+.3f}')
print(f"Cliff's delta = {cliffs_delta(F_v, M_v):+.3f}")
mw = mannwhitney(F_v, M_v)
print(f"Mann-Whitney p = {mw['p']:.4f}")
pm = perm_test_diff(F_v, M_v, stat=np.median)
print(f"Permutation p = {pm['p_two_sided']:.4f}")
print(power_note(len(F_v), len(M_v)))

print()
print(fa[['film_title','lead_gender','female_alter_betw_z']].round(3).to_string(index=False))

n_F=8  n_M=9
Median F=-0.188  Median M=-0.488
Cliff's delta = +0.194
Mann-Whitney p = 0.5414


Permutation p = 0.3020
With n_F=8 and n_M=9, Mann-Whitney U has very limited power. Minimum two-sided p achievable is 7.6e-06 only when the groups are perfectly separated; detection of a 'medium' effect (Cliff's δ≈0.33) is below ~30% at α=0.05.

               film_title lead_gender  female_alter_betw_z
             Mulan (1998)           F                0.797
       Toy Story 3 (2010)           M                0.379
            Onward (2020)           M                0.250
     Incredibles 2 (2018)           M                0.072
              Raya (2021)           F                0.054
Beauty & the Beast (1991)           F               -0.069
           Encanto (2021)           F               -0.161
            Frozen (2013)           F               -0.214
      Finding Nemo (2003)           M               -0.380
          Zootopia (2016)           F               -0.411
         Elemental (2023)           F               -0.448
         Toy Story (1995)           M         

---
## §B — Directed+Weighted Keystone Experiment

**Status: EXPERIMENT** — not part of the main analysis pipeline.

**Motivation:** The standard keystone uses *undirected* weighted betweenness. In a dialogue network, A speaks TO B is inherently asymmetric — directed betweenness distinguishes passive nodes (spoken to but don't initiate) from active bridges (channel dialogue flow). This experiment adds `ks_dirw` (directed + weighted betweenness) as a third comparison column.

**Question:** Does directed+weighted change the keystone identity, and does it change the gender-asymmetry finding from §5 of `09_analysis.ipynb`?

### 1. Load orchestrator keystone table + add directed+weighted column

In [3]:
ks = pd.read_csv(TBL / 'step2f_keystone_agreement.csv')
film_lookup = pd.read_csv(TBL / 'step2_per_film_method_comparison.csv')[['film_title', 'film_id']].drop_duplicates()
ks = ks.merge(film_lookup, on='film_title', how='left')
missing_ids = ks[ks['film_id'].isna()]['film_title'].tolist()
if missing_ids:
    raise ValueError(f'Missing film_id for: {missing_ids}')
print(f'Loaded {len(ks)} rows from step2f')

def keystone_dirw(film_id, p_id, remap, valid, id_to_name, gmap, min_w=3):
    e = pd.read_csv(DATA / film_id / 'network_edges.csv')
    G = nx.DiGraph()
    for _, r in e.iterrows():
        a=remap.get(r.speaker_clean,r.speaker_clean); b=remap.get(r.addressee_clean,r.addressee_clean)
        w=int(r.utterance_count)
        if a==b or a not in valid or b not in valid: continue
        if G.has_edge(a,b): G[a][b]['weight']+=w
        else: G.add_edge(a,b,weight=w)
    G.remove_edges_from([(a,b) for a,b,d in list(G.edges(data=True)) if d['weight']<min_w])
    for a,b,d in G.edges(data=True): G[a][b]['distance']=1.0/d['weight']
    if p_id not in G: return None, None
    btw=nx.betweenness_centrality(G,weight='distance',normalized=True)
    non_lead=[(n,v) for n,v in btw.items() if n!=p_id]
    if not non_lead: return None, None
    best=max(non_lead,key=lambda x:x[1])[0]
    return id_to_name.get(best,best), gmap.get(best)

dirw_names, dirw_genders, errors = [], [], []
for _, r in ks.iterrows():
    film_id = r['film_id']
    try:
        meta=load_character_meta(film_id); remap=build_id_remap(meta)
        gmap_f=gender_map(meta); valid=set(gmap_f.keys())
        p_id=protag_id(meta, r['protagonist'])
        id_to_name=dict(zip(meta['character_id'],meta['canonical_name']))
        name,g=keystone_dirw(film_id,p_id,remap,valid,id_to_name,gmap_f)
    except Exception as exc:
        errors.append((film_id, type(exc).__name__, str(exc)))
        name,g=None,None
    dirw_names.append(name); dirw_genders.append(g)

if errors:
    print('Directed+weighted failures:')
    for film_id, err_type, msg in errors:
        print(f'  {film_id}: {err_type}: {msg}')

ks['ks_dirw']=dirw_names; ks['ks_dirw_g']=dirw_genders
ks['addr_dirw_agree']=ks['keystone_addr']==ks['ks_dirw']
ks['flip_ad']=ks.apply(lambda r: r['keystone_addr_gender']!=r['ks_dirw_g']
                        if pd.notna(r['keystone_addr_gender']) and pd.notna(r['ks_dirw_g']) else False, axis=1)
print(ks[['film_title','lead_gender','keystone_addr','keystone_addr_gender','ks_dirw','ks_dirw_g','addr_dirw_agree','flip_ad']].to_string(index=False))
ks.to_csv(TBL / 'nb09b_keystone_three_methods.csv', index=False)


Loaded 17 rows from step2f


               film_title lead_gender keystone_addr keystone_addr_gender    ks_dirw ks_dirw_g  addr_dirw_agree  flip_ad
Beauty & the Beast (1991)           F         Beast                    M      Beast         M             True    False
             Mulan (1998)           F         Shang                    M      Shang         M             True    False
            Frozen (2013)           F          Elsa                    F       Elsa         F             True    False
        Inside Out (2015)           F         Anger                    M      Anger         M             True    False
          Zootopia (2016)           F          Bogo                    M       Bogo         M             True    False
           Encanto (2021)           F     Tio Bruno                    M    Isabela         F            False     True
              Raya (2021)           F          Sisu                    F       Sisu         F             True    False
         Elemental (2023)           F   

### 2. Summary

In [4]:
print(f'addr vs dirw agree: {ks.addr_dirw_agree.sum()}/{len(ks)} = {ks.addr_dirw_agree.mean():.1%}')
print()

for label,col in [('Addressee','keystone_addr_gender'),('Directed+Weighted','ks_dirw_g')]:
    ks['cross']=ks.apply(lambda r: r[col]!=r['lead_gender'] if pd.notna(r[col]) else False, axis=1)
    ct=ks.groupby('lead_gender')['cross'].agg(['sum','count'])
    ct.columns=['cross','n']; ct['pct']=(ct.cross/ct.n*100).round(1)
    print(f'{label}: cross-gender keystone'); print(ct); print()

for label,col_g in [('Addressee','keystone_addr_gender'),('Directed+Weighted','ks_dirw_g')]:
    ks['cross']=ks.apply(lambda r: r[col_g]!=r['lead_gender'] if pd.notna(r[col_g]) else False, axis=1)
    ct=pd.crosstab(ks['lead_gender'],ks['cross'])
    odds,p=stats.fisher_exact(ct.values)
    print(f'{label} Fisher: odds={odds:.3f}  p={p:.4f}', '✅' if p<0.05 else '')

addr vs dirw agree: 15/17 = 88.2%

Addressee: cross-gender keystone
             cross  n   pct
lead_gender                
F                6  8  75.0
M                2  9  22.2

Directed+Weighted: cross-gender keystone
             cross  n   pct
lead_gender                
F                5  8  62.5
M                2  9  22.2

Addressee Fisher: odds=0.095  p=0.0567 
Directed+Weighted Fisher: odds=0.171  p=0.1534 


### 3. Interpretation

addr vs dirw: **88.2% agreement (15/17)**. Only 2 keystones change under directed+weighted:
- *Encanto*: Tio Bruno -> Isabela
- *Toy Story 3*: Lotso -> Buzz

The directional version preserves the same broad asymmetry pattern, but weakens it: F-led films still have more cross-gender keystones than M-led films, but the F-led count drops from **6/8 (75.0%)** to **5/8 (62.5%)**. M-led films stay at **2/9 (22.2%)**. Fisher remains non-significant under both methods and becomes less suggestive with directionality (**p = 0.0567** addressee; **p = 0.1534** directed+weighted).

**Recommendation:** treat directed+weighted as a robustness/sensitivity check. The current undirected weighted pipeline is adequate for the main analysis because identity agreement is high and only one changed case flips keystone gender, but the directed version should be described as attenuating rather than fully confirming the gender-asymmetry signal.